# Models: Parametric vs DML

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.linear_model import ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import sys
from pathlib import Path
import statsmodels.api as sm
from econml.dml import LinearDML, CausalForestDML
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold


PROJECT_ROOT = Path.cwd().parent  # assumes notebooks/ is one level below root
sys.path.append(str(PROJECT_ROOT))

from src.load_data import load_feature
from src.results import sm_extract, econml_ate_row

c:\Users\danil\anaconda3\envs\vair312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = load_feature()
df.head()

,host_response_rate,host_acceptance_rate,host_is_superhost,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,accommodates,bathrooms,...,amenity_Dedicated workspace,amenity_Toaster,amenity_Freezer,amenity_Shower gel,amenity_First aid kit,amenity_Dining table,amenity_Cleaning products,amenity_Self check-in,amenity_Fire extinguisher,amenity_Long term stays allowed
0,1.00,0.96,1,1.098612,1.791759,2,1,1,1,1.0,...,1,1,1,1,1,1,0,1,1,1
1,0.88,0.88,1,1.386294,2.833213,3,1,1,6,2.0,...,1,1,1,0,0,1,1,0,0,1
2,1.00,0.98,0,1.386294,4.691348,2,1,1,4,1.0,...,0,0,0,0,0,0,0,0,0,0
3,1.00,0.91,0,0.693147,0.693147,3,1,1,5,1.5,...,1,0,0,0,0,0,0,1,1,0
4,1.00,1.00,1,1.098612,1.609438,2,1,1,2,0.0,...,1,0,0,0,1,0,0,1,1,0


In [3]:
df.dtypes.value_counts()

bool       131
int64       46
float64     20
Name: count, dtype: int64

In [4]:
df.select_dtypes(include="object").columns.tolist()

[]

## Parametric model

In [5]:
#first spec without borough
borough_columns = [i for i in df.columns if "borough" in i ]

In [6]:
#outcome and treatment variable
y = df["log_price"]
d =df["log_rivals_500m"]

#exclude and controls
exclude = {"log_price", "log_rivals_500m", "loc_fe", "log_rivals_c_sq", "log_rivals_c"}
X = df[[c for c in df.columns if (c not in exclude)]]


#controls and treatment
Z = pd.concat([d,X], axis=1)

#adding constant
Z = sm.add_constant(Z, has_constant="add")

#changing data types
y = y.astype(float)
Z = Z.astype(float)

Z.shape



(42898, 194)

In [7]:
#running model
ols2 = sm.OLS(y, Z).fit(
    cov_type="cluster",
    cov_kwds={"groups": df["loc_fe"]}
)

print(ols2.summary())

                            OLS Regression Results                            
Dep. Variable:              log_price   R-squared:                       0.727
Model:                            OLS   Adj. R-squared:                  0.725
Method:                 Least Squares   F-statistic:                 3.547e+04
Date:                Thu, 26 Feb 2026   Prob (F-statistic):          3.35e-197
Time:                        16:55:20   Log-Likelihood:                -20674.
No. Observations:               42898   AIC:                         4.174e+04
Df Residuals:                   42704   BIC:                         4.342e+04
Df Model:                         193                                         
Covariance Type:              cluster                                         
                                      coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
const     

c:\Users\danil\anaconda3\envs\vair312\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 193, but rank is 94
  warnings.warn('covariance of constraints does not have full '


In [8]:
beta = ols2.params["log_rivals_500m"]
se   = ols2.bse["log_rivals_500m"]
ci_l, ci_u = ols2.conf_int().loc["log_rivals_500m"].tolist()

print(beta, se, ci_l, ci_u)

0.018666948934831038 0.0121524492257893 -0.005151413871667652 0.04248531174132973


### Naive OLS, No FE

In [9]:
fe = [c for c in df.columns if c.startswith("fe_")]

In [10]:
#outcome and treatment variable
y_nl = df["log_price"]
d_nl =df["log_rivals_500m"]

#exclude and controls
exclude_nl = {"log_price", "log_rivals_500m", "loc_fe", "log_rivals_c", "log_rivals_c_sq"}
X_nl = df[[c for c in df.columns if (c not in exclude_nl) & (c not in fe) ]]


#controls and treatment
Z_nl = pd.concat([d_nl,X_nl], axis=1)

#adding constant
Z_nl = sm.add_constant(Z_nl, has_constant="add")

#changing data types
y_nl = y_nl.astype(float)
Z_nl = Z_nl.astype(float)

Z_nl.shape



(42898, 95)

In [11]:
print(f"{"log_rivals_c_sq" in Z_nl}")

False


In [12]:
#running model
ols1 = sm.OLS(y_nl, Z_nl).fit(
    cov_type="cluster",
    cov_kwds={"groups": df["loc_fe"]}
)

print(ols1.summary())

                            OLS Regression Results                            
Dep. Variable:              log_price   R-squared:                       0.706
Model:                            OLS   Adj. R-squared:                  0.705
Method:                 Least Squares   F-statistic:                 1.083e+04
Date:                Thu, 26 Feb 2026   Prob (F-statistic):          1.07e-171
Time:                        16:55:20   Log-Likelihood:                -22265.
No. Observations:               42898   AIC:                         4.472e+04
Df Residuals:                   42803   BIC:                         4.554e+04
Df Model:                          94                                         
Covariance Type:              cluster                                         
                                      coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
const     

### Double Lasso PDS

In [13]:
#setting up controls, treatment adn outcome

y_lasso = "log_price"
d_lasso = "log_rivals_500m"
fe_lasso = "loc_fe"

#fe columns
fe_cols = [c for c in df.columns if c.startswith("fe_")]

exclude = {y_lasso, d_lasso, "rivals_500m", "price", "id",
            fe_lasso, "log_rivals_c_sq", "log_rivals_c"}

# Candidate controls for selection (exclude FE dummies from selection step)
X_cand_cols = [c for c in df.columns if c not in exclude and c not in fe_cols]

# FE dummies forced-in at the final stage
X_force_cols = fe_cols

### Double selection

In [14]:
#Variabels for lasso
X_cand = df[X_cand_cols].values
Y = df[y_lasso].values.astype(np.float64)
T = df[d_lasso].values.astype(np.float64)

# ElasticNetCV with l1_ratio = 1 for lasso
enet = ElasticNetCV(
    l1_ratio=1.0,
    alphas=None,
    cv=5,
    random_state=0,
    max_iter=10000
)

# Pipeline: standardize then fit
y_selector = Pipeline([("scaler", StandardScaler()), ("enet", enet)])
t_selector = Pipeline([("scaler", StandardScaler()), ("enet", enet)])

y_selector.fit(X_cand, Y)
t_selector.fit(X_cand, T)

coef_y = y_selector.named_steps["enet"].coef_
coef_t = t_selector.named_steps["enet"].coef_

S_y = set(np.array(X_cand_cols)[coef_y != 0])
S_t = set(np.array(X_cand_cols)[coef_t != 0])
S_union = sorted(list(S_y.union(S_t)))

print("Selected for Y:", len(S_y))
print("Selected for T:", len(S_t))
print("Union selected:", len(S_union))


Selected for Y: 90
Selected for T: 90
Union selected: 90


In [15]:
removed = [c for c in X_cand_cols if c not in S_union]
removed

['amenity_Washer', 'amenity_Cooking basics', 'amenity_Hot water kettle']

In [16]:
difference = [c for c in S_y if c not in S_t]
difference

[]

### PDS

In [17]:
Z_cols = [d_lasso] + S_union + X_force_cols
Z_lasso = df[Z_cols].copy()
Z_lasso = Z_lasso.astype(float)
Z_lasso = sm.add_constant(Z_lasso, has_constant="add")

pds = sm.OLS(df[y_lasso].values, Z_lasso).fit(
    cov_type="cluster",
    cov_kwds={"groups": df[fe_lasso]}
)

print(pds.summary())
print("\nPDS beta:", pds.params[d_lasso], "SE:", pds.bse[d_lasso])

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.727
Model:                            OLS   Adj. R-squared:                  0.725
Method:                 Least Squares   F-statistic:                 2.269e+04
Date:                Thu, 26 Feb 2026   Prob (F-statistic):          2.25e-187
Time:                        16:55:22   Log-Likelihood:                -20688.
No. Observations:               42898   AIC:                         4.176e+04
Df Residuals:                   42707   BIC:                         4.341e+04
Df Model:                         190                                         
Covariance Type:              cluster                                         
                                      coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
const     

c:\Users\danil\anaconda3\envs\vair312\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 190, but rank is 91
  warnings.warn('covariance of constraints does not have full '


### DML

In [18]:
X_dml = df[X_cand_cols + X_force_cols].values
X_dml = X_dml.astype(np.float64)

dml = LinearDML(
    model_y=RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=20,
        n_jobs=-1,
        random_state=0
    ),
    model_t=RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=20,
        n_jobs=-1,
        random_state=0
    ),
    discrete_treatment=False,
    cv=KFold(n_splits=5, shuffle=True, random_state=0),
    random_state=0
)

dml.fit(Y, T, X=X_dml)


In [19]:
ate = dml.ate(X=X_dml)
print("Flexible DML ATE:", ate)


Flexible DML ATE: 0.009708573592984984


In [20]:
ci = dml.ate_interval(X=X_dml)
print("CI:", ci)

CI: (np.float64(-0.0007030316084277819), np.float64(0.020120178794397828))


### Causal forest

In [21]:
cf = CausalForestDML(
    model_y=RandomForestRegressor(n_estimators=300,
                                   min_samples_leaf=20, n_jobs=-1,
                                     random_state=0),
    model_t=RandomForestRegressor(n_estimators=300,
                                   min_samples_leaf=20, n_jobs=-1,
                                     random_state=0),
    n_estimators=1000,
    min_samples_leaf=50,
    random_state=42
)

cf.fit(Y, T, X=X_dml)

### Treatment heterogenity

In [22]:
cate = cf.effect(X_dml)
np.percentile(cate, [5, 25, 50, 75, 95])

array([-0.01877677,  0.01206159,  0.02750994,  0.0400592 ,  0.06169013])

In [23]:
dist_cate = px.histogram(
    x=cate,
    labels={"x": "TE"},
    title="Treatment Effect Distribution"
)
dist_cate.show()

## Results table

In [24]:
treat = "log_rivals_500m"

b1, se1, n1, r21 = sm_extract(ols1, treat)
b2, se2, n2, r22 = sm_extract(ols2, treat)
b3, se3, n3, r23 = sm_extract(pds, treat)

ate_dml, _, lo_dml, hi_dml = econml_ate_row(dml, X=X_dml)
ate_cf, _, lo_cf, hi_cf = econml_ate_row(cf, X=X_dml)
n_dml = len(X_dml)
n_cf = len(X_dml)

In [25]:
table = pd.DataFrame({
    "(1) OLS": [
        f"{b1:.4f}",
        f"({se1:.4f})",
        "Yes",
        "No",
        f"{n1}",
        f"{r21:.3f}"
    ],
    "(2) OLS + FE": [
        f"{b2:.4f}",
        f"({se2:.4f})",
        "Yes",
        "Yes",
        f"{n2}",
        f"{r22:.3f}"
    ],
    "(3) PDS": [
        f"{b3:.4f}",
        f"({se3:.4f})",
        "Selected via Lasso",
        "Yes",
        f"{n3}",
        f"{r23:.3f}"
    ],
    "(4) Linear DML": [
        f"{ate_dml:.4f}",
        f"[{lo_dml:.4f}, {hi_dml:.4f}]",
        "ML-adjusted",
        "Yes",
        f"{n_dml}",
        ""
    ],
    "(5) Causal Forest": [
        f"{ate_cf:.4f}",
        f"[{lo_cf:.4f}, {hi_cf:.4f}]",
        "ML-adjusted",
        "Yes",
        f"{n_cf}",
        ""
    ]
},
index=[
    "Log rivals (500m)",
    "",
    "Controls",
    "Location FE",
    "Observations",
    "R-squared"
])

table

,(1) OLS,(2) OLS + FE,(3) PDS,(4) Linear DML,(5) Causal Forest
Log rivals (500m),0.0722,0.0187,0.0186,0.0097,0.0252
,(0.0123),(0.0122),(0.0122),"[-0.0007, 0.0201]","[-0.0154, 0.0658]"
Controls,Yes,Yes,Selected via Lasso,ML-adjusted,ML-adjusted
Location FE,No,Yes,Yes,Yes,Yes
Observations,42898,42898,42898,42898,42898
R-squared,0.706,0.727,0.727,,
